# ⚽ Futsal AI Analysis — One-Click Video Generator
Upload your match video (e.g., 9-minute clip) and get the **final broadcast tactical video** (`broadcast_tactical.mp4`) with:
- Game-style player ellipses & jersey numbers (`#10`, `#7`, `#4`...)
- 2D Court Radar with Pitch Control (Voronoi cells)
- Dynamic Ball Possession ring, overhead pointer triangle, & trajectory trail
- Rolling substitutions stitched directly to jersey numbers
- SQLite match telemetry database (`telemetry.db`)

👉 Make sure **GPU** is selected: `Runtime → Change runtime type → T4 GPU`

### Step 1: Clone Repository & Install Dependencies

In [ ]:
!nvidia-smi -L
!git clone https://github.com/azamat-byte/futsal.git /content/futsal
%cd /content/futsal
!pip install -q -e ".[phase1,embeddings,colab]"
!pip install -q "supervision<=0.30.0"
!apt-get -qq install -y ffmpeg > /dev/null
print('Environment ready!')

### Step 2: Configure API Key & Match Video

In [ ]:
import os
from google.colab import userdata, drive

# Set your Roboflow API key (from Colab Secrets 🔑 or paste directly):
try:
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
except:
    os.environ['ROBOFLOW_API_KEY'] = 'YOUR_ROBOFLOW_API_KEY'

# Mount Google Drive to load your video and store the final output:
drive.mount('/content/drive')

# Path to your 9-minute match video in Google Drive:
INPUT_VIDEO = '/content/drive/MyDrive/futsal/9min_match.mp4'  # <-- Change this to your file!
OUTPUT_DIR = '/content/drive/MyDrive/futsal/final_output'

import cv2
cap = cv2.VideoCapture(INPUT_VIDEO)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f'Input Video: {INPUT_VIDEO}')
print(f'Total Frames: {total_frames} ({total_frames / max(fps, 1):.1f}s @ {fps:.1f} fps)')

### Step 3: Run the 1-Click Pipeline (Generates the Final Video)

In [ ]:
# Runs everything seamlessly: Detection -> Tracking -> Court Radar -> Possession -> Jersey OCR -> Broadcast Compositing
!python -m futsal.pipeline_runner \
    --video "{INPUT_VIDEO}" \
    --output "{OUTPUT_DIR}" \
    --stride-keypoints 30

print('\n======================================================')
print(f' FINAL VIDEO GENERATED:')
print(f' {OUTPUT_DIR}/broadcast_tactical.mp4')
print('======================================================')

### Step 4: Preview or Download the Final Video

In [ ]:
from google.colab import files

# You can download it directly to your computer (or access it in your Google Drive folder):
final_video_path = f'{OUTPUT_DIR}/broadcast_tactical.mp4'
files.download(final_video_path)

### (Optional) View Match Summary Telemetry

In [ ]:
import sqlite3, pandas as pd
from futsal.telemetry_sqlite import PLAYER_SUMMARY_BY_JERSEY_SQL

db_path = f'{OUTPUT_DIR}/telemetry.db'
con = sqlite3.connect(db_path)
print('=== Match Player Physical Summary (Grouped by Jersey Number) ===')
display(pd.read_sql(PLAYER_SUMMARY_BY_JERSEY_SQL, con))

print('\n=== Shot Events Detected ===')
display(pd.read_sql("SELECT * FROM match_events WHERE event_type='Shot Made'", con))